# CompactLLM — QLoRA fine-tune on Colab

Trains the Gemma-3 4B resume/JD relevance scorer, evaluates it against the base
model, and packages the adapter + benchmark for download.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**
2. Accept the Gemma license: https://huggingface.co/google/gemma-3-4b-it
3. Add Colab secrets (🔑 in the sidebar): `HF_TOKEN` (required, read scope),
   `WANDB_API_KEY` (optional — loss curves), `GROQ_API_KEY` (optional — the
   eval rationale judge)

Then: Runtime → Run all, and upload `train.jsonl` / `val.jsonl` / `test.jsonl`
when the upload cell prompts.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone --depth 1 https://github.com/aayush-arya/compact-llm.git
%cd compact-llm

In [ ]:
# Unsloth's pip package pulls a Colab-compatible torch/trl/peft stack.
# If it ever conflicts, fall back to: !pip install -q -r training/requirements.txt
!pip install -q unsloth
!pip install -q wandb httpx

In [ ]:
# Upload the split files (they're git-ignored). Pick train.jsonl, val.jsonl,
# and test.jsonl from your machine's data/processed/ folder.
import pathlib
from google.colab import files

pathlib.Path('data/processed').mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    pathlib.Path('data/processed', name).write_bytes(uploaded[name])
!wc -l data/processed/*.jsonl

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
for opt in ('WANDB_API_KEY', 'GROQ_API_KEY', 'CEREBRAS_API_KEY', 'GEMINI_API_KEY'):
    try:
        os.environ[opt] = userdata.get(opt)
    except Exception:
        pass

In [ ]:
# ~30-60 min on a T4. Writes outputs/adapter/ and outputs/merged/.
!python training/train_unsloth_qlora.py

In [ ]:
# Held-out eval: base zero-shot vs few-shot vs fine-tuned.
# --judge auto uses GROQ/CEREBRAS/GEMINI_API_KEY if set, else skips rationale grading.
!cd training && python eval_base_vs_finetuned.py --adapter_dir ../outputs/adapter --judge auto

In [ ]:
!zip -r compactllm-outputs.zip outputs/adapter outputs/merged docs/benchmark_results.json docs/benchmark_table.md
from google.colab import files
files.download('compactllm-outputs.zip')

## Back on your machine

```bash
unzip compactllm-outputs.zip        # -> outputs/adapter, outputs/merged, docs/benchmark_results.json
```

`outputs/` is git-ignored (push the adapter to a HF model repo for deployment).
`docs/benchmark_results.json` is committed — it lights up the Evaluation page
and the README table. Then run the app with `MODEL_BACKEND=transformers`.